# Red Traffic Sign Segmentation Pipeline

This pipeline leverages OpenCV and NumPy to isolate red traffic signs from cluttered real-world backgrounds. The process is broken into clear stages, each visualized side-by-side using Matplotlib, make it easier to follow the transformation from raw input to cleanly segmented output.

In [ ]:
# Import required libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

## 1. Image Loading and Preparation
The pipeline begins by reading all images from a designated local directory using OpenCV's `cv2.imread()`. Since OpenCV loads images in BGR format by default, we convert each image to RGB using `cv2.COLOR_BGR2RGB`. This conversion is essential because subsequent visualization with Matplotlib expects RGB ordering, ensuring that colors appear natural and accurate throughout the display steps.

In [ ]:
# Function to load an image and convert it from BGR to RGB
def load_image(filepath):
    img = cv2.imread(filepath)
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

## 2. Noise Reduction with Gaussian Blur
Before any color analysis, the RGB image is smoothed with a **3×3 Gaussian blur** (`cv2.GaussianBlur`). This lightweight filter reduces high-frequency sensor noise, minor scratches, and surface dust that could otherwise interfere with edge detection and color discrimination. Crucially, the small kernel size preserves the sharp outer boundaries and fine structural details of the traffic sign, maintaining critical geometric information for later processing stages.

In [ ]:
# Function to apply Gaussian Blur for noise reduction (using granular 3x3 kernel)
def apply_gaussian_blur(image, kernel_size=(3, 3)):
    return cv2.GaussianBlur(image, kernel_size, 0)

## 3. Generating the Red Difference Map
This step produces a grayscale intensity map that quantifies each pixel's "redness" while tolerating glare and illumination changes. The blurred RGB image is converted to HSV color space, and for each pixel, we compute its hue distance to pure red (accounting for red's wrap-around at 0° and 180°). This distance is transformed into a redness score (1.0 for pure red, 0.0 for hues 15° away). Simultaneously, the saturation channel acts as a glare filter: high-saturation pixels retain full weight, while low-saturation (washed-out) pixels are suppressed. The product of redness and saturation yields a robust map, which is then normalized to span the full 0–255 grayscale range.

In [ ]:
# Function to generate a normalized Red difference map using HSV color space
def get_red_diff_map(image):
    # Convert RGB to HSV
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    
    H = hsv[:,:,0].astype(np.float32)
    S = hsv[:,:,1].astype(np.float32)
    
    # In OpenCV HSV, Hue is 0-179. Pure red is around 0 and 180.
    # Calculate distance from pure red.
    dist = np.minimum(H, 180 - H)
    
    # Map distance to a redness score (0 to 1)
    # If dist is 0 (pure red), score is 1.0. If dist >= 15 (orange/yellow), score is 0.0.
    redness = np.maximum(0, 15 - dist) / 15.0
    
    # Multiply by Saturation to ignore white/gray pixels, but be tolerant of glare.
    # If S > 80, factor is 1.0. If S < 30, factor is 0.0. This prevents glare from breaking the ring.
    sat = np.clip((S - 30) / 50.0, 0, 1)
    
    # Combine to form the difference map
    diff = redness * sat * 255.0
    
    # Normalize to 0-255 to ensure Otsu's thresholding has a full range to work with
    diff = cv2.normalize(diff, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX)
    
    return diff.astype(np.uint8)

## 4. Adaptive Color Thresholding
Instead of relying on a fixed intensity cutoff, the pipeline employs **Otsu's binarization** (`cv2.THRESH_BINARY + cv2.THRESH_OTSU`) on the red difference map. Otsu's method automatically determines the optimal threshold by minimizing intra-class variance, effectively separating red-signal pixels from background noise. This adaptive approach makes the pipeline resilient to varying lighting conditions, overexposure, and shadows—situations where a manual threshold would frequently fail. The output is a clean binary mask with white pixels marking candidate red regions.

In [ ]:
# Function to perform adaptive thresholding (Otsu's method) on the difference map
def adaptive_color_threshold(diff_map):
    # Otsu's thresholding dynamically finds the best threshold value
    ret, mask = cv2.threshold(diff_map, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return mask

## 5. Morphological Cleaning
The binary mask undergoes two morphological operations using elliptical structuring elements, which naturally respect the curved contours of circular and triangular signs. First, **morphological opening** (erosion followed by dilation) with a 3×3 kernel removes isolated specks of noise and breaks accidental connections to red background objects like brick walls or car taillights. Next, **morphological closing** (dilation followed by erosion) with a 5×5 kernel bridges small gaps and repairs fractured segments along the sign's red border caused by glare, fading, or physical wear, resulting in a cohesive, solid ring.

In [ ]:
# Function to apply Morphological Cleaning (Opening first to detach noise, Closing to fill gaps)
def clean_morphology(mask):
    # Use elliptical kernels for natural rounding around circular traffic signs
    open_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    close_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    
    # Opening FIRST: Erosion followed by Dilation. This breaks thin bridges connecting background noise to the sign.
    opening = cv2.morphologyEx(mask, cv2.MORPH_OPEN, open_kernel)
    
    # Closing NEXT: Dilation followed by Erosion. An elliptical 5x5 kernel smoothly bridges gaps in the sign ring.
    closing = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, close_kernel)
    return closing

## 6. Mask Extraction and Interior Filling
With a clean border mask in hand, the pipeline isolates the largest connected red component using 4-connected component analysis (`cv2.connectedComponentsWithStats`). This step specifically avoids grouping diagonally touching noise, ensuring only the primary sign is selected. The external contour of the chosen component is then extracted and filled completely (`cv2.FILLED`), generating a filled mask that covers the entire sign area—including any black symbols, numbers, or white backgrounds inside the red border. Finally, a bitwise AND between the original RGB image and this filled mask yields the fully segmented traffic sign on a black background, preserving all original color information without distortion.

In [ ]:
# Function to apply the cleaned mask to the original image, filling internal area without distortion
def extract_and_mask(image, cleaned_mask):
    # Use 4-connectivity to prevent diagonal bridges from attaching background red objects (like awnings) to the sign
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(cleaned_mask, connectivity=4)
    
    # If only background is found
    if num_labels <= 1:
        return np.zeros_like(image)
        
    # Find the label with the largest area (excluding the background which is label 0)
    largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    
    # Create a new mask containing only the largest connected component
    largest_mask = np.uint8(labels == largest_label) * 255
    
    # Find external contours of the isolated sign component
    contours, _ = cv2.findContours(largest_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled_mask = np.zeros_like(largest_mask)
    if contours:
        # Draw all external contours filled (-1 thickness) to include the interior area of the sign naturally
        cv2.drawContours(filled_mask, contours, -1, 255, thickness=cv2.FILLED)
    else:
        filled_mask = largest_mask
        
    # Bitwise-AND the original image with the filled mask
    result = cv2.bitwise_and(image, image, mask=filled_mask)
    return result

## 7. Batch Processing and Visual Comparison
The pipeline automatically processes every image found in the input folder, generating a comprehensive 6-column Matplotlib grid for each. This side-by-side visualization displays the **original image**, **Gaussian-blurred version**, **red difference map**, **Otsu threshold result**, **morphologically cleaned mask**, and the final **segmented RGB sign**. This structured layout allows users to intuitively inspect the effect of each stage, verify the algorithm's decisions, and quickly identify any images that may require parameter tuning—making the pipeline not only functional but also transparent and educational.

In [ ]:
# Process multiple images and plot original vs segmented images side-by-side
# Get all image paths from the Inputs folder
image_paths = glob.glob("Inputs/Red signs/*.*")

if not image_paths:
    print("No images found in Inputs/Red signs/.")
else:
    # Set up matplotlib figure (6 columns for full intermediate visualization)
    num_images = len(image_paths)
    fig, axes = plt.subplots(num_images, 6, figsize=(24, 4 * num_images))
    
    # Handle the case where there is only one image (axes is 1D)
    if num_images == 1:
        axes = [axes]
        
    for i, path in enumerate(image_paths):
        # 1. Load image
        img = load_image(path)
        if img is None:
            continue
            
        # 2. Preprocess: Blur
        blurred_img = apply_gaussian_blur(img)
        
        # 3. Preprocess: Red-Blue Diff
        diff_map = get_red_diff_map(blurred_img)
        
        # 4. Process: Threshold
        binary_mask = adaptive_color_threshold(diff_map)
        
        # 5. Output: Clean
        cleaned_mask = clean_morphology(binary_mask)
        
        # 6. Output: Extract
        segmented_img = extract_and_mask(img, cleaned_mask)
        
        # Plotting 6 columns side-by-side
        row_axes = axes[i] if num_images > 1 else axes
        
        row_axes[0].imshow(img)
        row_axes[0].set_title(f"1. Original\n{os.path.basename(path)}", fontsize=10)
        row_axes[0].axis('off')
        
        row_axes[1].imshow(blurred_img)
        row_axes[1].set_title("2. Gaussian Blur", fontsize=10)
        row_axes[1].axis('off')
        
        row_axes[2].imshow(diff_map, cmap='gray')
        row_axes[2].set_title("3. Red Diff Map", fontsize=10)
        row_axes[2].axis('off')
        
        row_axes[3].imshow(binary_mask, cmap='gray')
        row_axes[3].set_title("4. Otsu Threshold", fontsize=10)
        row_axes[3].axis('off')
        
        row_axes[4].imshow(cleaned_mask, cmap='gray')
        row_axes[4].set_title("5. Morphology & Fill", fontsize=10)
        row_axes[4].axis('off')
        
        row_axes[5].imshow(segmented_img)
        row_axes[5].set_title("6. Segmented RGB", fontsize=10)
        row_axes[5].axis('off')

    plt.tight_layout()
    plt.show()